In [1]:
!rm -rf /content/EmbodiedLLM/ /content/Evo_1/
!git clone https://github.com/PrithviTM-glitch/EmbodiedLLM.git

!cp -r /content/EmbodiedLLM/MultipleHooksStudy/models/Evo1StateExperiments ./Evo_1

!rm -rf MultipleHooksStudy test .claude .github .vscode metaworld_* state_* README.md

Cloning into 'EmbodiedLLM'...
remote: Enumerating objects: 1457, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 1457 (delta 0), reused 2 (delta 0), pack-reused 1425 (from 1)
Receiving objects: 100% (1457/1457), 151.69 MiB | 14.26 MiB/s, done.
Resolving deltas: 100% (377/377), done.


In [2]:
!pip install -r Evo_1/Evo_1/requirements.txt
!MAX_JOBS=64 pip install -v flash-attn --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 101.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 137.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 162.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 152.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 115.3 MB/s eta 0:00:00


In [3]:
%%bash
pip install mujoco
pip install metaworld
pip install websockets
pip install opencv-python
pip install packaging
pip install huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 152.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 83.8 MB/s eta 0:00:00


In [4]:
%%bash

mkdir Evo1_training_dataset/
cd Evo1_training_dataset/
GIT_LFS_SKIP_SMUDGE=1 git clone https://huggingface.co/datasets/MINT-SJTU/Evo1_MetaWorld_Dataset Metaworld
cd Metaworld/
git lfs pull

Cloning into 'Metaworld'...


In [5]:
%%bash

pip install torch torchvision accelerate transformers
pip install wandb swanlab einops timm
pip install pandas pyarrow av

In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/Evo1_experiments/metaworld/exp1_position', exist_ok=True)
os.makedirs('/content/drive/MyDrive/Evo1_experiments/metaworld/exp2a_strainA', exist_ok=True)
os.makedirs('/content/drive/MyDrive/Evo1_experiments/metaworld/exp2c_strainC', exist_ok=True)
os.makedirs('/content/drive/MyDrive/Evo1_experiments/cache/metaworld', exist_ok=True)

Mounted at /content/drive


In [7]:
# Already Done Metaworld Data exists
# !cp -r /content/Evo1_training_dataset /content/drive/MyDrive/Evo1_training_dataset

In [8]:
import sys
sys.path.append('/content/Evo_1/Evo_1/model/action_head')

import torch
from state_encoder import build_encoder
from flow_matching import FlowmatchingActionHead
from types import SimpleNamespace

# --- Test 1: Encoder shapes ---
encoder_config = {
    'features': ['position', 'velocity', 'acceleration', 'trace', 'deviation'],
    'history_len': 5,
    'embedding_strain': 'C',
    'state_dim': 7,
    'embed_dim': 896,
    'hidden_dim': 1024,
    'trace_decay': 0.9,
}
encoder = build_encoder(encoder_config)

B, k = 2, 5
state_history = torch.randn(B, k+1, 7)
tokens = encoder(state_history)

assert tokens.shape == (B, 3*k+2, 896), f"Wrong shape: {tokens.shape}"
print(f"✓ Encoder output: {tokens.shape}  expected ({B}, {3*k+2}, 896)")

# --- Test 2: num_tokens property ---
assert encoder.num_tokens == 3*k+2
print(f"✓ num_tokens: {encoder.num_tokens}  expected {3*k+2}")

# --- Test 3: All three strains ---
for strain in ['A', 'B', 'C']:
    cfg = {**encoder_config, 'embedding_strain': strain}
    enc = build_encoder(cfg)
    out = enc(state_history)
    assert out.shape == (B, 3*k+2, 896)
    print(f"✓ Strain {strain} output: {out.shape}")

# --- Test 4: Position only (Exp 1) ---
pos_config = {**encoder_config, 'features': ['position'], 'embedding_strain': 'none'}
pos_encoder = build_encoder(pos_config)
pos_tokens = pos_encoder(state_history)
assert pos_tokens.shape == (B, k+1, 896), f"Wrong shape: {pos_tokens.shape}"
print(f"✓ Position only output: {pos_tokens.shape}  expected ({B}, {k+1}, 896)")

# --- Test 5: Full forward pass with dummy VLM tokens ---
N = 100
fused_tokens        = torch.randn(B, N, 896)
actions_gt          = torch.randn(B, 16, 7)
action_mask_train   = torch.ones(B, 16, 7)    # [B, H, per_action_dim] for training
action_mask_infer   = torch.ones(B, 7)        # [B, per_action_dim] for inference

head = FlowmatchingActionHead(config=SimpleNamespace(
    embed_dim=896,
    hidden_dim=1024,
    action_dim=16*7,
    horizon=16,
    per_action_dim=7,
    num_heads=8,
    num_layers=2,
    dropout=0.0,
    num_inference_timesteps=10,
    num_categories=1,
    encoder_config=encoder_config,
))

pred_velocity, noise = head(
    fused_tokens,
    state=state_history,
    actions_gt=actions_gt,
    action_mask=action_mask_train,
)
assert pred_velocity.shape == (B, 16*7), f"Wrong shape: {pred_velocity.shape}"
print(f"✓ pred_velocity: {pred_velocity.shape}  expected ({B}, {16*7})")

# --- Test 6: Inference path (action_mask forwarded fix) ---
action = head.get_action(
    fused_tokens,
    state=state_history,
    action_mask=action_mask_infer,
)
assert action.shape == (B, 16*7), f"Wrong shape: {action.shape}"
print(f"✓ get_action: {action.shape}  expected ({B}, {16*7})")

# --- Test 7: freeze_pretrained_params ---
encoder.freeze_pretrained_params()
frozen    = [n for n, p in encoder.named_parameters() if not p.requires_grad]
trainable = [n for n, p in encoder.named_parameters() if p.requires_grad]
print(f"✓ Frozen params: {len(frozen)}")
print(f"✓ Trainable params (decay logits only): {trainable}")
assert all('decay' in n for n in trainable), "Only decay_logits should be trainable after freeze"
print("✓ freeze_pretrained_params correct")

print("\n✓ All shape tests passed")

✓ Encoder output: torch.Size([2, 17, 896])  expected (2, 17, 896)
✓ num_tokens: 17  expected 17
✓ Strain A output: torch.Size([2, 17, 896])
✓ Strain B output: torch.Size([2, 17, 896])
✓ Strain C output: torch.Size([2, 17, 896])
✓ Position only output: torch.Size([2, 6, 896])  expected (2, 6, 896)
num_inference_timesteps 10
✓ pred_velocity: torch.Size([2, 112])  expected (2, 112)
action_mask shape: torch.Size([2, 7])
one sample action_mask: tensor([1., 1., 1., 1., 1., 1., 1.])
action shape: torch.Size([2, 112])
one sample action: tensor([-0.6906,  0.7797,  0.9969,  0.2517,  0.4154,  0.9574,  0.3347,  0.5823,
         0.3258,  0.9712,  0.2968,  0.3459,  0.5175,  0.3525, -0.5532, -0.5941,
        -0.5836, -0.3145,  0.9150, -0.8718,  0.5162, -0.7440, -0.6379,  0.3249,
        -0.1938,  0.1110, -0.5832,  0.7603,  0.5240,  0.2822,  0.3462, -0.7063,
        -0.8643,  0.3834,  0.5036, -0.4408,  0.8701,  0.1884,  0.7435, -0.4109,
        -0.8920,  0.5938,  0.2610, -0.8956,  0.1754,  0.8724,  0.

In [9]:
wandb_api_key = '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'

In [10]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tmprithvi (tmprithvi-nanyang-technological-university-singapore) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [11]:
# 1. Confirm all paths actually exist
import os
paths = [
    "/content/Evo_1/Evo_1/scripts/train.py",
    "/content/Evo_1/Evo_1/ds_config.json",
    "/content/Evo_1/Evo_1/dataset/metaworld_config.yaml",
    "/content/drive/MyDrive/Evo1_experiments/cache/metaworld",
]
for p in paths:
    status = "✓" if os.path.exists(p) else "✗ MISSING"
    print(f"{status}  {p}")

✓  /content/Evo_1/Evo_1/scripts/train.py
✓  /content/Evo_1/Evo_1/ds_config.json
✓  /content/Evo_1/Evo_1/dataset/metaworld_config.yaml
✓  /content/drive/MyDrive/Evo1_experiments/cache/metaworld


In [12]:
!sed -i 's|/content/Evo1_training_dataset/MetaWorld|/content/Evo1_training_dataset/Metaworld|g' \
    /content/Evo_1/Evo_1/dataset/metaworld_config.yaml

In [13]:
import wandb
import subprocess
import os

FIXED_ARGS = [
    "--action_head", "flowmatching",
    "--use_augmentation",
    "--lr", "1e-5",
    "--dropout", "0.2",
    "--weight_decay", "1e-3",
    "--batch_size", "16",
    "--image_size", "448",
    "--max_steps", "1",
    "--log_interval", "10",
    "--ckpt_interval", "1000",
    "--warmup_steps", "200",
    "--grad_clip_norm", "1.0",
    "--num_layers", "8",
    "--horizon", "50",
    "--per_action_dim", "24",
    "--state_dim", "24",
    "--finetune_action_head",
    "--vlm_name", "OpenGVLab/InternVL3-1B",
    "--dataset_config_path", "/content/Evo_1/Evo_1/dataset/metaworld_config.yaml",
    "--cache_dir", "/content/drive/MyDrive/Evo1_experiments/cache/metaworld",
    "--history_len", "5",
    "--embedding_strain", "B",
    "--features", "position", "velocity", "acceleration", "trace", "deviation",
    "--trace_decay", "0.9",
    "--eval_interval", "9999",
    "--disable_swanlab",
    "--wandb_project", "evo1_metaworld",
]

sweep_config = {
    "method": "bayes",
    "metric": {"name": "phase0/final_recon", "goal": "minimize"},
    "parameters": {
        "pretrain_steps": {"values": [500, 1000, 2000, 3000]},
        "pretrain_lr":    {"values": [1e-2, 1e-3, 1e-4]},
        "lambda_orth":    {"values": [0.001, 0.01, 0.1]},
    }
}

os.makedirs("/content/sweep_logs", exist_ok=True)

def train_sweep():
    run = wandb.init()
    pretrain_steps = run.config.pretrain_steps
    pretrain_lr    = run.config.pretrain_lr
    lambda_orth    = run.config.lambda_orth
    run_id         = run.id
    run.finish()

    env = os.environ.copy()
    env["WANDB_RUN_ID"]  = run_id
    env["WANDB_RESUME"]  = "allow"
    env["WANDB_PROJECT"] = "evo1_metaworld"
    env["WANDB_ENTITY"]  = "tmprithvi-nanyang-technological-university-singapore"
    env["WANDB_MODE"]    = "online"

    cmd = [
        "accelerate", "launch",
        "--num_processes", "1",
        "--num_machines", "1",
        "--mixed_precision", "bf16",
        "--dynamo_backend", "no",
        "--deepspeed_config_file", "/content/Evo_1/Evo_1/ds_config.json",
        "/content/Evo_1/Evo_1/scripts/train.py",
    ] + FIXED_ARGS + [
        "--pretrain_steps", str(pretrain_steps),
        "--pretrain_lr",    str(pretrain_lr),
        "--lambda_orth",    str(lambda_orth),
        "--run_name",       f"sweep_{run_id}",
        "--save_dir",       f"/content/drive/MyDrive/Evo1_experiments/evo1_sweep/{run_id}",
    ]

    log_path = f"/content/sweep_logs/{run_id}.log"
    with open(log_path, "w") as log_file:
        process = subprocess.Popen(
            cmd,
            env=env,
            stdout=log_file,
            stderr=log_file,
        )
        process.wait()

    if process.returncode != 0:
        # Print last 50 lines of log on failure so you can see the error
        with open(log_path, "r") as f:
            lines = f.readlines()
        print(f"\n=== Run {run_id} FAILED (exit {process.returncode}) ===")
        print("".join(lines[-50:]))
        raise RuntimeError(f"train.py exited with code {process.returncode}")

os.environ["WANDB_AGENT_MAX_INITIAL_FAILURES"] = "20"
sweep_id = wandb.sweep(sweep_config, project="evo1_metaworld")
wandb.agent(sweep_id, function=train_sweep, count=8)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Create sweep with ID: kpaq8jl3
Sweep URL: https://wandb.ai/tmprithvi-nanyang-technological-university-singapore/evo1_metaworld/sweeps/kpaq8jl3


wandb: Agent Starting Run: 3b81iufc with config:
wandb: 	lambda_orth: 0.1
wandb: 	pretrain_lr: 0.0001
wandb: 	pretrain_steps: 2000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: tmprithvi (tmprithvi-nanyang-technological-university-singapore) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: 01oxylxt with config:
wandb: 	lambda_orth: 0.01
wandb: 	pretrain_lr: 0.01
wandb: 	pretrain_steps: 3000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


wandb: Agent Starting Run: ge3ob6ot with config:
wandb: 	lambda_orth: 0.001
wandb: 	pretrain_lr: 0.001
wandb: 	pretrain_steps: 1000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


wandb: Agent Starting Run: o82l7u1u with config:
wandb: 	lambda_orth: 0.1
wandb: 	pretrain_lr: 0.0001
wandb: 	pretrain_steps: 2000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


wandb: Agent Starting Run: y4i7k6o6 with config:
wandb: 	lambda_orth: 0.01
wandb: 	pretrain_lr: 0.001
wandb: 	pretrain_steps: 1000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: bimk8oaq with config:
wandb: 	lambda_orth: 0.1
wandb: 	pretrain_lr: 0.0001
wandb: 	pretrain_steps: 2000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


wandb: Agent Starting Run: z6l42hqj with config:
wandb: 	lambda_orth: 0.01
wandb: 	pretrain_lr: 0.0001
wandb: 	pretrain_steps: 1000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


wandb: Agent Starting Run: 1jbo1lvz with config:
wandb: 	lambda_orth: 0.01
wandb: 	pretrain_lr: 0.001
wandb: 	pretrain_steps: 1000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


In [14]:
import glob
logs = glob.glob('/content/drive/MyDrive/Evo1_experiments/evo1_sweep/*/train_log_*.log')
latest = max(logs, key=__import__('os').path.getmtime)
print(f"Reading: {latest}")
!tail -50 {latest}

Reading: /content/drive/MyDrive/Evo1_experiments/evo1_sweep/1jbo1lvz/train_log_20260402_072614.log
2026-04-02 07:49:34,254 [INFO] Frozen state_encoder.mlp.0.weight                              | shape: (1024, 24)           |   0.02M
2026-04-02 07:49:34,254 [INFO] Frozen state_encoder.mlp.0.bias                                | shape: (1024,)              |   0.00M
2026-04-02 07:49:34,254 [INFO] Frozen state_encoder.mlp.2.weight                              | shape: (896, 1024)          |   0.92M
2026-04-02 07:49:34,255 [INFO] Frozen state_encoder.mlp.2.bias                                | shape: (896,)               |   0.00M
2026-04-02 07:49:34,255 [INFO] Frozen state_encoder.layer_norm.weight                         | shape: (896,)               |   0.00M
2026-04-02 07:49:34,255 [INFO] Frozen state_encoder.layer_norm.bias                           | shape: (896,)               |   0.00M
2026-04-02 07:49:34,256 [INFO] Trainable action_encoder.W1.linear.weight                         

In [16]:
!mkdir /content/drive/MyDrive/Evo1_experiments/evo1_sweep_logs

In [17]:
!cp -r /content/sweep_logs/* /content/drive/MyDrive/Evo1_experiments/evo1_sweep_logs

---
## Full Training Runs — Exp1, Exp2A, Exp2B, Exp2C

Each experiment has two stages:
- **Stage 1** (5 000 steps): action-head only, Phase 0 pretraining active
- **Stage 2** (80 000 steps): full VLM + action-head finetune, resumes from Stage 1 `step_5000` checkpoint

Note: Stage 2 at 80 k steps on an L4 is ~450 h per run — use `--resume` to continue across Colab sessions.

### Exp1 — Position history, Strain `none`

In [ ]:
%%bash
cd /content/Evo_1/Evo_1
accelerate launch \
  --num_processes 1 --num_machines 1 \
  --mixed_precision bf16 --dynamo_backend no \
  --deepspeed_config_file ds_config.json \
  scripts/train.py \
  --run_name Evo1_metaworld_exp1_stage1 \
  --wandb_project evo1_metaworld \
  --action_head flowmatching \
  --use_augmentation \
  --lr 1e-5 --dropout 0.2 --weight_decay 1e-3 \
  --batch_size 16 --image_size 448 \
  --max_steps 5000 --warmup_steps 1000 \
  --log_interval 10 --ckpt_interval 2500 \
  --eval_interval 500 \
  --grad_clip_norm 1.0 --num_layers 8 \
  --horizon 50 --per_action_dim 24 --state_dim 24 \
  --finetune_action_head \
  --vlm_name OpenGVLab/InternVL3-1B \
  --dataset_config_path /content/Evo_1/Evo_1/dataset/metaworld_config.yaml \
  --cache_dir /content/drive/MyDrive/Evo1_experiments/cache/metaworld \
  --save_dir /content/drive/MyDrive/Evo1_experiments/metaworld/exp1/stage1 \
  --history_len 5 \
  --features position \
  --embedding_strain none \
  --pretrain_steps 1000 --pretrain_lr 0.0001 --lambda_orth 0.1 \
  --trace_decay 0.9 \
  --disable_swanlab

In [ ]:
%%bash
# Exp1 Stage 2 — resume from step_5000, full VLM finetune
# Confirm step_5000 dir exists before running:
ls /content/drive/MyDrive/Evo1_experiments/metaworld/exp1/stage1/step_5000

In [ ]:
%%bash
cd /content/Evo_1/Evo_1
accelerate launch \
  --num_processes 1 --num_machines 1 \
  --mixed_precision bf16 --dynamo_backend no \
  --deepspeed_config_file ds_config.json \
  scripts/train.py \
  --run_name Evo1_metaworld_exp1_stage2 \
  --wandb_project evo1_metaworld \
  --action_head flowmatching \
  --use_augmentation \
  --lr 1e-5 --dropout 0.2 --weight_decay 1e-3 \
  --batch_size 16 --image_size 448 \
  --max_steps 80000 --warmup_steps 1000 \
  --log_interval 10 --ckpt_interval 2500 \
  --eval_interval 500 \
  --grad_clip_norm 1.0 --num_layers 8 \
  --horizon 50 --per_action_dim 24 --state_dim 24 \
  --finetune_vlm --finetune_action_head \
  --vlm_name OpenGVLab/InternVL3-1B \
  --dataset_config_path /content/Evo_1/Evo_1/dataset/metaworld_config.yaml \
  --cache_dir /content/drive/MyDrive/Evo1_experiments/cache/metaworld \
  --save_dir /content/drive/MyDrive/Evo1_experiments/metaworld/exp1/stage2 \
  --history_len 5 \
  --features position \
  --embedding_strain none \
  --pretrain_steps 1000 --pretrain_lr 0.0001 --lambda_orth 0.1 \
  --trace_decay 0.9 \
  --skip_pretrain \
  --resume_model_only \
  --resume --resume_path /content/drive/MyDrive/Evo1_experiments/metaworld/exp1/stage1/step_5000 \
  --disable_swanlab

### Exp2A — Full features, Strain `A`

In [ ]:
%%bash
cd /content/Evo_1/Evo_1
accelerate launch \
  --num_processes 1 --num_machines 1 \
  --mixed_precision bf16 --dynamo_backend no \
  --deepspeed_config_file ds_config.json \
  scripts/train.py \
  --run_name Evo1_metaworld_exp2A_stage1 \
  --wandb_project evo1_metaworld \
  --action_head flowmatching \
  --use_augmentation \
  --lr 1e-5 --dropout 0.2 --weight_decay 1e-3 \
  --batch_size 16 --image_size 448 \
  --max_steps 5000 --warmup_steps 1000 \
  --log_interval 10 --ckpt_interval 2500 \
  --eval_interval 500 \
  --grad_clip_norm 1.0 --num_layers 8 \
  --horizon 50 --per_action_dim 24 --state_dim 24 \
  --finetune_action_head \
  --vlm_name OpenGVLab/InternVL3-1B \
  --dataset_config_path /content/Evo_1/Evo_1/dataset/metaworld_config.yaml \
  --cache_dir /content/drive/MyDrive/Evo1_experiments/cache/metaworld \
  --save_dir /content/drive/MyDrive/Evo1_experiments/metaworld/exp2A/stage1 \
  --history_len 5 \
  --features position velocity acceleration trace deviation \
  --embedding_strain A \
  --pretrain_steps 1000 --pretrain_lr 0.0001 --lambda_orth 0.1 \
  --trace_decay 0.9 \
  --disable_swanlab

In [ ]:
%%bash
ls /content/drive/MyDrive/Evo1_experiments/metaworld/exp2A/stage1/step_5000

In [ ]:
%%bash
cd /content/Evo_1/Evo_1
accelerate launch \
  --num_processes 1 --num_machines 1 \
  --mixed_precision bf16 --dynamo_backend no \
  --deepspeed_config_file ds_config.json \
  scripts/train.py \
  --run_name Evo1_metaworld_exp2A_stage2 \
  --wandb_project evo1_metaworld \
  --action_head flowmatching \
  --use_augmentation \
  --lr 1e-5 --dropout 0.2 --weight_decay 1e-3 \
  --batch_size 16 --image_size 448 \
  --max_steps 80000 --warmup_steps 1000 \
  --log_interval 10 --ckpt_interval 2500 \
  --eval_interval 500 \
  --grad_clip_norm 1.0 --num_layers 8 \
  --horizon 50 --per_action_dim 24 --state_dim 24 \
  --finetune_vlm --finetune_action_head \
  --vlm_name OpenGVLab/InternVL3-1B \
  --dataset_config_path /content/Evo_1/Evo_1/dataset/metaworld_config.yaml \
  --cache_dir /content/drive/MyDrive/Evo1_experiments/cache/metaworld \
  --save_dir /content/drive/MyDrive/Evo1_experiments/metaworld/exp2A/stage2 \
  --history_len 5 \
  --features position velocity acceleration trace deviation \
  --embedding_strain A \
  --pretrain_steps 1000 --pretrain_lr 0.0001 --lambda_orth 0.1 \
  --trace_decay 0.9 \
  --skip_pretrain \
  --resume_model_only \
  --resume --resume_path /content/drive/MyDrive/Evo1_experiments/metaworld/exp2A/stage1/step_5000 \
  --disable_swanlab

### Exp2B — Full features, Strain `B`

In [ ]:
%%bash
cd /content/Evo_1/Evo_1
accelerate launch \
  --num_processes 1 --num_machines 1 \
  --mixed_precision bf16 --dynamo_backend no \
  --deepspeed_config_file ds_config.json \
  scripts/train.py \
  --run_name Evo1_metaworld_exp2B_stage1 \
  --wandb_project evo1_metaworld \
  --action_head flowmatching \
  --use_augmentation \
  --lr 1e-5 --dropout 0.2 --weight_decay 1e-3 \
  --batch_size 16 --image_size 448 \
  --max_steps 5000 --warmup_steps 1000 \
  --log_interval 10 --ckpt_interval 2500 \
  --eval_interval 500 \
  --grad_clip_norm 1.0 --num_layers 8 \
  --horizon 50 --per_action_dim 24 --state_dim 24 \
  --finetune_action_head \
  --vlm_name OpenGVLab/InternVL3-1B \
  --dataset_config_path /content/Evo_1/Evo_1/dataset/metaworld_config.yaml \
  --cache_dir /content/drive/MyDrive/Evo1_experiments/cache/metaworld \
  --save_dir /content/drive/MyDrive/Evo1_experiments/metaworld/exp2B/stage1 \
  --history_len 5 \
  --features position velocity acceleration trace deviation \
  --embedding_strain B \
  --pretrain_steps 1000 --pretrain_lr 0.0001 --lambda_orth 0.1 \
  --trace_decay 0.9 \
  --disable_swanlab

In [ ]:
%%bash
ls /content/drive/MyDrive/Evo1_experiments/metaworld/exp2B/stage1/step_5000

In [ ]:
%%bash
cd /content/Evo_1/Evo_1
accelerate launch \
  --num_processes 1 --num_machines 1 \
  --mixed_precision bf16 --dynamo_backend no \
  --deepspeed_config_file ds_config.json \
  scripts/train.py \
  --run_name Evo1_metaworld_exp2B_stage2 \
  --wandb_project evo1_metaworld \
  --action_head flowmatching \
  --use_augmentation \
  --lr 1e-5 --dropout 0.2 --weight_decay 1e-3 \
  --batch_size 16 --image_size 448 \
  --max_steps 80000 --warmup_steps 1000 \
  --log_interval 10 --ckpt_interval 2500 \
  --eval_interval 500 \
  --grad_clip_norm 1.0 --num_layers 8 \
  --horizon 50 --per_action_dim 24 --state_dim 24 \
  --finetune_vlm --finetune_action_head \
  --vlm_name OpenGVLab/InternVL3-1B \
  --dataset_config_path /content/Evo_1/Evo_1/dataset/metaworld_config.yaml \
  --cache_dir /content/drive/MyDrive/Evo1_experiments/cache/metaworld \
  --save_dir /content/drive/MyDrive/Evo1_experiments/metaworld/exp2B/stage2 \
  --history_len 5 \
  --features position velocity acceleration trace deviation \
  --embedding_strain B \
  --pretrain_steps 1000 --pretrain_lr 0.0001 --lambda_orth 0.1 \
  --trace_decay 0.9 \
  --skip_pretrain \
  --resume_model_only \
  --resume --resume_path /content/drive/MyDrive/Evo1_experiments/metaworld/exp2B/stage1/step_5000 \
  --disable_swanlab

### Exp2C — Full features, Strain `C`

In [ ]:
%%bash
cd /content/Evo_1/Evo_1
accelerate launch \
  --num_processes 1 --num_machines 1 \
  --mixed_precision bf16 --dynamo_backend no \
  --deepspeed_config_file ds_config.json \
  scripts/train.py \
  --run_name Evo1_metaworld_exp2C_stage1 \
  --wandb_project evo1_metaworld \
  --action_head flowmatching \
  --use_augmentation \
  --lr 1e-5 --dropout 0.2 --weight_decay 1e-3 \
  --batch_size 16 --image_size 448 \
  --max_steps 5000 --warmup_steps 1000 \
  --log_interval 10 --ckpt_interval 2500 \
  --eval_interval 500 \
  --grad_clip_norm 1.0 --num_layers 8 \
  --horizon 50 --per_action_dim 24 --state_dim 24 \
  --finetune_action_head \
  --vlm_name OpenGVLab/InternVL3-1B \
  --dataset_config_path /content/Evo_1/Evo_1/dataset/metaworld_config.yaml \
  --cache_dir /content/drive/MyDrive/Evo1_experiments/cache/metaworld \
  --save_dir /content/drive/MyDrive/Evo1_experiments/metaworld/exp2C/stage1 \
  --history_len 5 \
  --features position velocity acceleration trace deviation \
  --embedding_strain C \
  --pretrain_steps 1000 --pretrain_lr 0.0001 --lambda_orth 0.1 \
  --trace_decay 0.9 \
  --disable_swanlab

In [ ]:
%%bash
ls /content/drive/MyDrive/Evo1_experiments/metaworld/exp2C/stage1/step_5000

In [ ]:
%%bash
cd /content/Evo_1/Evo_1
accelerate launch \
  --num_processes 1 --num_machines 1 \
  --mixed_precision bf16 --dynamo_backend no \
  --deepspeed_config_file ds_config.json \
  scripts/train.py \
  --run_name Evo1_metaworld_exp2C_stage2 \
  --wandb_project evo1_metaworld \
  --action_head flowmatching \
  --use_augmentation \
  --lr 1e-5 --dropout 0.2 --weight_decay 1e-3 \
  --batch_size 16 --image_size 448 \
  --max_steps 80000 --warmup_steps 1000 \
  --log_interval 10 --ckpt_interval 2500 \
  --eval_interval 500 \
  --grad_clip_norm 1.0 --num_layers 8 \
  --horizon 50 --per_action_dim 24 --state_dim 24 \
  --finetune_vlm --finetune_action_head \
  --vlm_name OpenGVLab/InternVL3-1B \
  --dataset_config_path /content/Evo_1/Evo_1/dataset/metaworld_config.yaml \
  --cache_dir /content/drive/MyDrive/Evo1_experiments/cache/metaworld \
  --save_dir /content/drive/MyDrive/Evo1_experiments/metaworld/exp2C/stage2 \
  --history_len 5 \
  --features position velocity acceleration trace deviation \
  --embedding_strain C \
  --pretrain_steps 1000 --pretrain_lr 0.0001 --lambda_orth 0.1 \
  --trace_decay 0.9 \
  --skip_pretrain \
  --resume_model_only \
  --resume --resume_path /content/drive/MyDrive/Evo1_experiments/metaworld/exp2C/stage1/step_5000 \
  --disable_swanlab